# Suite GEN — Sampling and generators

Samples are specified exactly on the random source's 32-bit words, so both implementations draw the same values from the
same seed. `Generate` streams new objects drawn from a distribution: each step chooses a case by weight, and builds what
the case's predicate requires by equality.

In [ ]:
import math
from collections import Counter

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Constraints as C, Distributions as D, Generators as G, Predicates as P, Queries as Q, Sampling as M, Validators as V
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S, Stores
from support import Contact, Directory, Phone, book, listed, raises


class Words:
    """A random source of the given words, for exact cases."""

    def __init__(self, *words):
        self.words = list(words)

    def next_u32(self):
        return self.words.pop(0)

    def split(self, key):
        return self


MAX = 2**32 - 1


APerson = {"person": Contact}
person = E.variable("person")
HasName = (P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())


def named(weight, name):
    return lambda a: a.weight(weight).requires(HasName(person, name))


Names = P.OfPredicate.Builder().name("Names").symbols(APerson).requires(D.Choices.Builder().arms(
    named(10, "alice"), named(5, "ben"), named(15, "chermon"), named(7, "davi")).decreasing().create()).create()

## GEN-01 · Samples are specified on the source's words

In [ ]:
r = Stores.PCG32(42, 54)
assert [M.uniform(r) for _ in range(3)] == [0.6303102186438938, 0.7270080560068604, 0.7486033647998483]
r = Stores.PCG32(42, 54)
assert [M.below(r, 6) for _ in range(8)] == [3, 3, 2, 1, 1, 4, 5, 3]
r = Stores.PCG32(42, 54)
assert [M.below(r, 2**40) for _ in range(3)] == [788047328265, 208370070163, 325543878766]  # two words each
r = Stores.PCG32(42, 54)
assert [M.weighted(r, [10.0, 5.0, 15.0, 7.0]) for _ in range(10)] == [2, 2, 2, 2, 3, 0, 3, 0, 2, 2]
assert M.uniform(Words(0, 0)) == 0.0 and M.uniform(Words(2**32 - 1, 2**32 - 1)) == 1 - 2**-53  # the ends
assert M.below(Words(2**32 - 1, 5), 3) == 2  # 2**32 - 1 is past the last multiple of 3: rejected, then 5 % 3
assert M.below(Words(7), 1) == 0 and M.weighted(Words(0, 0), [0.0, 2.0, 0.0]) == 1
assert M.weighted(Words(2**32 - 1, 2**32 - 1), [1.0, 2.0, 0.0]) == 1  # the last positive weight, at the top
with raises(ValueError, match="below needs a positive int, got 0"):
    M.below(Words(), 0)
with raises(ValueError, match="a weight must be a finite float, not negative, got -1.0"):
    M.weighted(Words(), [1.0, -1.0])
with raises(ValueError, match="a weight must be a finite float, not negative, got inf"):
    M.weighted(Words(), [1.0, float("inf")])
with raises(ValueError, match="a weight must be a finite float, not negative, got 1"):
    M.weighted(Words(), [1])  # an int: Python only, since TypeScript numbers are floats
with raises(ValueError, match="no weight is positive"):
    M.weighted(Words(), [0.0, 0.0])

## GEN-02 · Generate chooses arms by weight and sets what their equalities require

In [ ]:
store = book()
generated = G.Generate(store, Names, Stores.PCG32(42))
first = [next(generated)["person"].name for _ in range(12)]
assert first == ["chermon", "alice", "chermon", "ben", "davi", "davi", "chermon", "alice", "alice", "ben", "alice", "chermon"]
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(42)), range(12))] == first  # the same seed
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(7)), range(12))] != first
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(42).split("other")), range(12))] != first
names = Counter(next(generated)["person"].name for _ in range(3700))
assert all(abs(names[n] - 100 * w) < 100 for n, w in (("alice", 10), ("ben", 5), ("chermon", 15), ("davi", 7))), names
assert iter(generated) is generated and (generated.steps, generated.rejected) == (3712, 0)
Fixed = P.OfPredicate.Builder().symbols(APerson).requires(E.literal(30).eq(person.age).and_(HasName(person, "cy"))).requires(
    person.name.eq(person.name), E.literal(1).eq(1)).create()  # either side of ==, through and and applications
made = next(G.Generate(book(), Fixed, Stores.PCG32(1)))["person"]
assert (made.name, made.age) == ("cy", 30)  # the other equalities set nothing

## GEN-03 · A predicate that cannot be generated is refused

In [ ]:
Grown = P.OfPredicate.Builder().symbols(APerson).requires(person.age.ge(18)).create()
with raises(ValueError, match="the predicate cannot be generated from its equalities and choices: what they build does not satisfy it"):
    next(G.Generate(book(), Grown, Stores.PCG32(42)))
Shadowed = P.OfPredicate.Builder().symbols(APerson).requires(D.Choices.Builder().arms(
    lambda a: a.weight(1e-9).requires(person.has("name")), named(1, "alice")).decreasing().create()).create()
with raises(ValueError, match="the predicate cannot be generated from its equalities and choices"):
    next(G.Generate(book(), Shadowed, Stores.PCG32(42)))  # alice falls under the first arm, not the one chosen
with raises(ValueError, match="the predicate cannot be drawn from: rule: a choices needs an arm"):
    G.Generate(book(), P.OfPredicate.Builder().symbols(APerson).requires(D.Choices.Builder().create()).create(), Stores.PCG32(42))
with raises(ValueError, match="the predicate cannot be drawn from: a predicate with parameters is drawn from where it is applied"):
    G.Generate(book(), HasName, Stores.PCG32(42))
Wrong = P.OfPredicate.Builder().symbols(APerson).requires(
    D.Normal.Builder().symbol("v").mean(0).deviation(1).requires(person.age.eq(E.variable("v"))).create()).create()
with raises(ValueError, match="the predicate cannot be drawn from: person.age is int, but its distribution gives float"):
    G.Generate(book(), Wrong, Stores.PCG32(42))

## GEN-04 · Generated data is ordinary data: listed, validated, queried and written

In [ ]:
store = book()
people = [m["person"] for m, _ in zip(G.Generate(store, Names, Stores.PCG32(42)), range(6))]
assert list(store.extent("Contact")) == []  # transient until linked to the store's data
listed(store, *people)
assert len(list(store.extent("Contact"))) == 6
NamedChermon = P.OfPredicate.Builder().name("NamedChermon").symbols({"the": Contact}).requires(HasName(E.variable("the"), "chermon")).create()
chermons = [m["the"] for m in Q.Scan(store).select(NamedChermon)]
assert chermons == [p for p in people if p.name == "chermon"] and chermons
problems = V.Validate(store, [NamedChermon]).Reachable(Directory, store.singleton("book.Directory"))
assert len(problems) == 6 - len(chermons) and all(p.endswith("'NamedChermon' does not hold") for p in problems)
assert '"name": "ben"' in JSON.ToJSON(store).Reachable(Directory, store.singleton("book.Directory"))

## GEN-05 · Values are sampled exactly from the source's words

In [ ]:
r = Stores.PCG32(42, 54)
assert [M.between(r, -1.0, 1.0) for _ in range(2)] == [0.2606204372877876, 0.4540161120137207]
assert [M.normal(r, 10.0, 2.0) for _ in range(2)] == [11.674255764143341, 10.036009018664974]
assert [M.poisson(r, 2.5) for _ in range(4)] == [5, 1, 3, 4] and M.poisson(r, 1234.5) == 1233  # over 500: a sum
assert [M.geometric(r, 0.25) for _ in range(4)] == [0, 1, 2, 1] and M.geometric(Words(), 1.0) == 0
assert [M.log(x) for x in (1.0, 2.0, 0.5, 10.0, 1e-300, 5e-324)] == [
    0.0, 0.6931471805599453, -0.6931471805599453, 2.302585092994046, -690.7755278982137, -744.4400719213812]
assert [M.exp(x) for x in (0.0, 1.0, -1.0, 0.1, 700.0, -740.0)] == [
    1.0, 2.7182818284590455, 0.36787944117144233, 1.1051709180756477, 1.0142320547350045e+304, 4.2e-322]
assert math.isnan(M.log(-1.0)) and math.isnan(M.log(math.nan)) and M.log(0.0) == -math.inf and M.log(math.inf) == math.inf
assert math.isnan(M.exp(math.nan)) and M.exp(710.0) == math.inf and M.exp(-746.0) == 0.0
HALF, THREE_QUARTERS = 2**31, 3 * 2**30
assert M.normal(Words(0, 0, 0, 0, HALF, 0, HALF, 0, THREE_QUARTERS, 0, THREE_QUARTERS, 0), 0.0, 1.0) == 0.5 * math.sqrt(
    -2 * M.log(0.5) / 0.5)  # s = 2, then s = 0, are drawn again
assert M.poisson(Words(MAX, MAX), 500.0) == 1583  # u near 1: the sum stops when its terms underflow
for call, message in ((lambda: M.between(Words(), 1.0, 0.0), "a uniform's low must not exceed its high, got 1.0 and 0.0"),
                      (lambda: M.between(Words(), math.inf, 1.0), "a uniform's low must be finite, got inf"),
                      (lambda: M.normal(Words(), 0.0, -1.0), "a normal's deviation must not be negative, got -1.0"),
                      (lambda: M.normal(Words(), math.nan, 1.0), "a normal's mean must be finite, got nan"),
                      (lambda: M.poisson(Words(), -1.0), "a Poisson's rate must not be negative, got -1.0"),
                      (lambda: M.poisson(Words(), math.inf), "a Poisson's rate must be finite, got inf"),
                      (lambda: M.geometric(Words(), 0.0), "a geometric's probability must be in (0, 1], got 0.0")):
    with raises(ValueError, match=message):
        call()

## GEN-06 · Distributions draw what the equalities leave open, redrawn until the predicate holds

In [ ]:
def aged(distribution):
    return distribution.symbol("age").requires(lambda person, age: person.age == age).create()


Ages = P.OfPredicate.Builder().symbols(APerson).requires(D.Choices.Builder().arms(
    lambda a: a.weight(3).requires(HasName(person, "senior"), aged(D.Normal.Builder().mean(70).deviation(8).rounded()),
                                   person.age.ge(65)),
    lambda a: a.weight(7).requires(HasName(person, "adult"), aged(D.Uniform.Builder().low(18).high(64)))).decreasing().create()
).create()
generation = G.Generate(book(), Ages, Stores.PCG32(42))
people = [next(generation)["person"] for _ in range(1000)]
seniors, adults = [p.age for p in people if p.name == "senior"], [p.age for p in people if p.name == "adult"]
assert abs(len(seniors) - 300) < 60 and min(seniors) >= 65 and all(18 <= age <= 64 for age in adults)  # the arm holds
assert generation.steps == 1000 and generation.rejected > 0  # seniors under 65, redrawn
Item = S.OfObject.Builder().name("Item").ref().properties(
    lambda p: p.name("a").of(lambda t: t.as_native(int)), lambda p: p.name("b").of(lambda t: t.as_native(int)),
    lambda p: p.name("c").of(lambda t: t.as_native(float)), lambda p: p.name("d").of(lambda t: t.as_native(float))).create()
items = Proxies.OfStore()
items.register(Item)
item, a = E.variable("item"), E.variable("a")
Pairs = P.OfPredicate.Builder().symbols({"item": Item}).requires(
    D.Uniform.Builder().symbol("a").low(1).high(6).requires(item.a.eq(a), D.Uniform.Builder().symbol("b").low(a).high(a)
                                                            .requires(item.b.eq(E.variable("b"))).create()).create(),
    D.Uniform.Builder().symbol("c").low(0.0).high(1.0).requires(item.c.eq(E.variable("c"))).create(),  # floats in [0, 1)
    D.Normal.Builder().symbol("d").mean(0.5).deviation(0.0).requires(item.d.eq(E.variable("d"))).create()).create()
pairs = [next(G.Generate(items, Pairs, Stores.PCG32(seed)))["item"] for seed in range(6)]  # b reads a, as drawn
assert all(i.b == i.a and 1 <= i.a <= 6 and 0 <= i.c < 1 and i.d == 0.5 for i in pairs) and len({i.a for i in pairs}) > 1
Fixed = P.OfPredicate.Builder().symbols({"item": Item}).requires(item.a.eq(4), item.b.eq(7)).requires(
    D.Uniform.Builder().symbol("a").low(1).high(6).requires(item.a.eq(a)).create()).create()
fixed = next(G.Generate(items, Fixed, Stores.PCG32(1)))["item"]
assert (fixed.a, fixed.b) == (4, 7)  # what an equality sets first is not drawn
Between = P.OfPredicate.Builder().name("Between").symbols({"item": Item}).parameters(lambda p: p.name("low")).requires(
    D.Uniform.Builder().symbol("a").low(E.variable("low")).high(E.variable("low")).requires(item.a.eq(a)).create()).create()
Applied = P.OfPredicate.Builder().symbols({"item": Item}).requires(Between(item, E.literal(3).add(2))).create()
assert next(G.Generate(items, Applied, Stores.PCG32(1)))["item"].a == 5  # an argument, evaluated for the walk
Never = P.OfPredicate.Builder().symbols({"item": Item}).requires(
    D.Uniform.Builder().symbol("a").low(1).high(6).requires(item.a.eq(a)).create(), item.a.gt(6)).create()
never = G.Generate(items, Never, Stores.PCG32(1))
with raises(ValueError, match="the predicate cannot be generated: none of its 100 attempts satisfies it"):
    next(never)
assert never.rejected == G.ATTEMPTS and never.steps == 0
Reversed = P.OfPredicate.Builder().symbols({"item": Item}).requires(
    D.Uniform.Builder().symbol("a").low(6).high(1).requires(item.a.eq(a)).create()).create()
with raises(ValueError, match="a uniform's low must not exceed its high, got 6 and 1"):
    next(G.Generate(items, Reversed, Stores.PCG32(1)))
Text = P.OfPredicate.Builder().symbols({"item": Item}).requires(
    D.Poisson.Builder().symbol("a").rate("x").requires(item.a.eq(a)).create()).create()
with raises(TypeError, match="a Poisson's rate must be a number, got str"):
    next(G.Generate(items, Text, Stores.PCG32(1)))
Picked = P.OfPredicate.Builder().symbols({"item": Item}).requires(
    D.Categorical.Builder().symbol("a").option(1, 2).option(3, 5).requires(item.a.eq(a)).create(),
    D.Geometric.Builder().symbol("b").probability(1.0).requires(item.b.eq(E.variable("b"))).create()).create()
picked = [next(G.Generate(items, Picked, Stores.PCG32(seed)))["item"] for seed in range(8)]
assert {i.a for i in picked} == {2, 5} and {i.b for i in picked} == {0}

## GEN-07 · Generating at the seams: links, two symbols, read-back patterns and negations

In [ ]:
p = E.variable("p")
owns = P.Exists(lambda q: q.symbols({"p": Phone}).requires(P.Contains(person.phones, lambda e: e.phone == p)))
Linked = P.OfPredicate.Builder().symbols(APerson).requires(owns).create()
with raises(ValueError, match="the predicate cannot be generated from its equalities and choices"):
    next(G.Generate(book(), Linked, Stores.PCG32(1)))  # a generator builds no links: open, in PATTERNS.md
c = E.variable("c")
Two = P.OfPredicate.Builder().symbols({"c": Contact, "p": Phone}).requires(
    D.Uniform.Builder().symbol("n").low(1).high(3).requires(c.age.eq(E.variable("n"))).create(),
    D.Categorical.Builder().symbol("v").option(1, "1").option(1, "2").requires(p.number.eq(E.variable("v"))).create()).create()
store = book()
pairs = [next(G.Generate(store, Two, Stores.PCG32(seed))) for seed in range(6)]
assert all(1 <= m["c"].age <= 3 and m["p"].number in ("1", "2") for m in pairs)  # each symbol its own object
listed(store, *(m["c"] for m in pairs))
Aged = P.OfPredicate.Builder().symbols({"c": Contact}).requires(c.age.ge(1)).create()
assert next(G.Sample(store, Aged, Stores.PCG32(1)))["c"] in [m["c"] for m in pairs]  # and sampled, once listed
schemas = C.OfStore(book())
back = JSON.FromJSON(schemas).Reachable(P.OfPredicate.Schema, JSON.ToJSON(schemas).Reachable(P.OfPredicate.Schema, Names))
assert [m["person"].name for m, _ in zip(G.Generate(book(), back, Stores.PCG32(42)), range(12))] == [
    m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(42)), range(12))]  # read back, the same data
Not = P.OfPredicate.Builder().symbols(APerson).requires(HasName(person, "zed")).forbids(Names.rule).create()
assert next(G.Generate(book(), Not, Stores.PCG32(1)))["person"].name == "zed"  # a negation is checked, not walked